In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import os


def excel_to_sqlite(excel_file, db_path="data.db"):
    """Convert uploaded Excel into SQLite DB (one table per sheet)."""

    if os.path.exists(db_path):
        os.remove(db_path)

    engine = create_engine(f"sqlite:///{db_path}")

    xls = pd.ExcelFile(excel_file)

    for sheet in xls.sheet_names:
        df = xls.parse(sheet)
        table_name = sheet.replace(" ", "_")
        df.to_sql(table_name, engine, index=False, if_exists="replace")

    return db_path

db_path = excel_to_sqlite(excel_file="DAILY ACTIVITY REPORT.xlsx")


In [ ]:
from langchain_community.utilities import SQLDatabase
from langchain_classic.chains import create_sql_query_chain
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_mistralai import ChatMistralAI
from langchain_core.runnables import RunnableLambda
import re

os.environ["MISTRAL_API_KEY"] = ""

db = SQLDatabase.from_uri(f"sqlite:///{db_path}")

llm = ChatMistralAI(model="mistral-large-latest")

sql_chain = create_sql_query_chain(llm, db)

answer_prompt = PromptTemplate.from_template(
    """
Analyze the Data inside the DB based on the question

Question: {question}
SQL Query: {query}
SQL Result: {result}
Answer:
"""
)

def extract_sql(text: str) -> str:
    """Extract raw SQL from LLM output."""
    match = re.search(r"SELECT.*?;", text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(0)
    raise ValueError("No SQL query found in LLM output")

def run_sql(query: str):
    return db.run(query)

chain = (
    RunnablePassthrough.assign(raw_query=sql_chain)
    .assign(query=RunnableLambda(lambda x: extract_sql(x["raw_query"])))
    .assign(result=lambda x: run_sql(x["query"]))
    | answer_prompt
    | llm
    | StrOutputParser()
)

response = chain.invoke({"question": "Read and summarize all the data present, and come up with a good analytics"})
print(response)


In [ ]:
import pandas as pd
import os
from langchain_experimental.tools import PythonAstREPLTool
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import ChatPromptTemplate

file_path = "DAILY ACTIVITY REPORT.xlsx"
df_raw = pd.read_excel(file_path)

headers = df_raw.iloc[11].values
df = df_raw.iloc[12:].copy()
df.columns = headers
df.reset_index(drop=True, inplace=True)

llm = ChatMistralAI(model="mistral-large-latest")

tools = [PythonAstREPLTool(locals={"df": df})]

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful data analyst. Use Python to analyze the dataframe df.",
        ),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)



In [ ]:
response = agent_executor.invoke(
    {"input": "Summarize the container data and total payload"}
)

print(type(response))
print(response)

In [ ]:
import os
import pandas as pd
from mistralai import Mistral
from dotenv import load_dotenv

load_dotenv()
api_key = ""

client = Mistral(api_key=api_key)


def excel_to_text(file_path: str, max_rows: int = 200) -> str:
    df = pd.read_excel(file_path)

    preview = df.head(max_rows).to_csv(index=False)

    return f"""
Dataset shape: {df.shape}
Columns: {list(df.columns)}

Preview:
{preview}
"""


def get_insights(data_text: str) -> str:
    prompt = f"""
You are a senior data analyst.

Provide:
- Trends
- Patterns
- Outliers
- Business insights
- Recommended actions

Dataset:
{data_text}
"""

    response = client.chat.complete(
        model="mistral-large-latest",
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=0.2,
    )

    return response.choices[0].message.content


def main():
    data_text = excel_to_text("DAILY ACTIVITY REPORT.xlsx")
    insights = get_insights(data_text)

    print("\n===== INSIGHTS =====\n")
    print(insights)


if __name__ == "__main__":
    main()
